# 🧠 Agentic Psychometric Analysis 

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://www.macalester.edu/psychology/wp-content/uploads/sites/585/2019/11/psychology-feature.jpg"> 
</p>
</div>

## Description : 

Agentic Psychometric Analysis is a modular, agent-based AI system designed to perform comprehensive psychometric evaluations using specialized language model agents.

Each agent focuses on a unique psychological dimension — including:

- **Psychological assessment**

- **Response analysis**

- **Mindfulness techniques** 

- **Cognitive behavioral strategies**  

- **Positive psychology practices**  

- **Stress management tools**  

- **Ethical considerations**  

The system allows for highly personalized analysis by processing user context and responses through purpose-built agents, enabling a holistic and multi-perspective psychological insight pipeline.

Built with extensibility and clarity in mind, it is ideal for:

- **Mental health research**

- **Personal development applications**  

- **Experimental AI-based therapy simulations**

## Step 1: Environment Setup and Installation

This cell handles initial setup for the notebook:

- Installs dependencies from `requirements/psycho_analysis.requirements.txt`.

- Retries installation up to 3 times on failure.

- Loads environment variables from `.env` using `python-dotenv`.

- Ensures `OPENAI_API_KEY` is set before continuing.

After setup, it clears the output and confirms success.


In [2]:
# Boilerplate: This block goes into every notebook.
# It sets up the environment, installs the requirements, and checks for the required environment variables.

from IPython.display import clear_output
from dotenv import load_dotenv
import os

requirements_installed = False
max_retries = 3
retries = 0
REQUIRED_ENV_VARS = ["OPENAI_API_KEY"]
PROJECT_NAME = "agentic_psychometric_analysis"
REQUIREMENTS_FILE = f"{PROJECT_NAME}.requirements.txt"


def install_requirements():
    """Installs the requirements from requirements.txt file"""
    global requirements_installed, retries, max_retries
    if requirements_installed:
        print("Requirements already installed.")
        return

    print("Installing requirements...")
    install_status = os.system(f"pip install -r requirements/{REQUIREMENTS_FILE}")
    if install_status == 0:
        print("Requirements installed successfully.")
        requirements_installed = True
    else:
        print("Failed to install requirements.")
        if retries < max_retries:
            print("Retrying...")
            retries += 1
            return install_requirements()
        exit(1)
    return


def setup_env():
    """Sets up the environment variables"""

    def check_env(env_var):
        value = os.getenv(env_var)
        if value is None:
            print(f"Please set the {env_var} environment variable.")
            exit(1)
        else:
            print(f"{env_var} is set.")

    load_dotenv(override=True)

    variables_to_check = REQUIRED_ENV_VARS

    for var in variables_to_check:
        check_env(var)


install_requirements()
clear_output()
setup_env()
print("🚀 Setup complete. Continue to the next cell.")

OPENAI_API_KEY is set.
🚀 Setup complete. Continue to the next cell.


## Step 2: Set OpenAI API Key for Agents

This step retrieves the `OPENAI_API_KEY` from environment variables  

and sets it as the default key for use by agent utilities via `set_default_openai_key()`.


In [3]:
from agents import set_default_openai_key
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

set_default_openai_key(OPENAI_API_KEY)

## Step 3: (Optional) List Available OpenAI Models

Use this cell to print available OpenAI model names if the dropdown  

in your UI or IntelliSense fails to load them correctly.  

Set `enable_cell = True` to activate this workaround.  

**Note:** This step is not required to run the agent.


In [4]:
## NOTE: Use this cell to get the model names incase the dropdown does not work.
## My IntelliSense was showing error classes in the models dropdown when creating the agent.
## This is a workaround to get the model names.
## This cell is not required to run the agent.

enable_cell = False

from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

if enable_cell:
    models = client.models.list()
    for model in models:
        print(model.id)

## Step 4: Define the Psychometrics Agent Architecture

This step defines the `PsychometricsAgent` class, which wraps around the `Agent` and `Runner` to provide a reusable interface for running structured psychometric tasks.


Key Features:

- Accepts `BaseModel` inputs and outputs for strong typing.

- Uses an instruction-driven LLM agent backend.

- Includes a `from_config()` and `from_batch_config()` method for easy initialization from dictionaries.

- Supports verbosity for debugging purposes.

Also includes the `PsychometricAgentContainer` for organizing multiple agents in batch mode.


In [45]:
from agents import Agent, Runner
from pydantic import BaseModel
from typing import List


class PsychometricAgentContainer:
    name: str
    agent: "PsychometricsAgent"


class PsychometricsAgent:
    """
    Abstract base class for psychometrics agents.
    """

    def __init__(
        self,
        name: str,
        instructions: str,
        model: str,
        input_type: BaseModel,
        output_type: BaseModel,
        verbose: bool = False,
    ):
        """
        Initialize the agent with a model and verbosity level.
        """
        self.model = model
        self.verbose = verbose
        self.name = name
        self.instructions = instructions
        self.output_type = output_type
        self.input_type = input_type

        self.agent = Agent(
            name=name,
            instructions=instructions,
            model=model,
            output_type=output_type,
        )

    async def run(self, input: BaseModel) -> BaseModel:
        """
        Run the agent with the given input.

        Args:
            input (BaseModel): The input to the agent.
        Returns:
            BaseModel: The output of the agent.
        """
        if self.verbose:
            print(
                f"Running {self.name} agent with input: {input.model_dump_json(indent=4)}"
            )

        prompt = f"""Input:\n'''{input.model_dump_json(indent=4)}'''"""
        response = await Runner.run(
            self.agent,
            input=prompt,
        )
        output = response.final_output

        if self.verbose:
            print(f"Output:\n'''{output.model_dump_json(indent=4)}'''")

        return output

    @classmethod
    def from_config(cls, config: dict):
        """
        Create an agent from a config dictionary.
        """
        return cls(
            name=config["name"],
            instructions=config["instructions"],
            model=config["model"],
            input_type=config["input_type"],
            output_type=config["output_type"],
            verbose=config.get("verbose", False),
        )

    @classmethod
    def from_batch_config(cls, config: dict) -> List["PsychometricAgentContainer"]:
        """
        Create an agent from a batch config dictionary.
        """
        agents: List[PsychometricAgentContainer] = []
        for key, value in config.items():
            agent = cls.from_config(value)
            agents.append(
                PsychometricAgentContainer(
                    name=key,
                    agent=agent,
                    input_type=agent.input_type,
                    output_type=agent.output_type,
                )
            )
        return agents

## Step 5: Define Agent IO Models and Agent Configuration

This step defines the **Pydantic input/output schemas** for each specialized agent:  

- `AssessmentAgent`, `ResponseAnalyzerAgent`, `MindfulnessAgent`, etc.

Each agent:

- Takes structured input such as personal info, context, and responses.

- Produces structured output like questions, summaries, scores, or techniques.

It also defines:

- **`CombinedInput` and `CombinedOutput`** schemas for pipeline chaining.

- **Agent constants** for model variants.

- A centralized **`AGENT_CONFIG`** dictionary to configure each agent's:

  - Name
  
  - Instructions
  
  - Model
  
  - Input/Output Types
  
  - Verbose flag


In [43]:
from pydantic import BaseModel
from typing import List

LLM_MODEL_GPT_4_1_NANO = "gpt-4.1-nano"
LLM_MODEL_GPT_4_1_MINI = "gpt-4.1-mini"
LLM_MODEL_GPT_4_1 = "gpt-4.1"
GLOBAL_VERBOSE_ENABLED = True


class AssessmentInput(BaseModel):
    personal_info: List[str]
    context: str


class AssessmentOutput(BaseModel):
    questions: List[str]


class ResponseAnalyzerInput(BaseModel):
    personal_info: List[str]
    context: str
    questions: List[str]
    responses: List[str]


class ResponseAnalyzerOutput(BaseModel):
    analysis: str
    summary: str
    recommendations: List[str]
    score: float


class MindfulnessInput(BaseModel):
    personal_info: List[str]
    context: str
    questions: List[str]
    responses: List[str]


class MindfulnessOutput(BaseModel):
    mindfulness_exercises: List[str]
    techniques: List[str]


class CognitiveBehavioralInput(BaseModel):
    personal_info: List[str]
    context: str
    questions: List[str]
    responses: List[str]


class CognitiveBehavioralOutput(BaseModel):
    cognitive_techniques: List[str]
    exercises: List[str]


class PositivePsychologyInput(BaseModel):
    personal_info: List[str]
    context: str
    questions: List[str]
    responses: List[str]


class PositivePsychologyOutput(BaseModel):
    positive_techniques: List[str]
    exercises: List[str]


class StressManagementInput(BaseModel):
    personal_info: List[str]
    context: str
    questions: List[str]
    responses: List[str]


class StressManagementOutput(BaseModel):
    stress_management_techniques: List[str]
    exercises: List[str]


class EthicalInput(BaseModel):
    personal_info: List[str]
    context: str
    questions: List[str]
    responses: List[str]


class EthicalOutput(BaseModel):
    ethical_guidelines: List[str]
    considerations: List[str]


class CombinedInput(BaseModel):
    assesment_input: AssessmentInput
    response_analyzer_input: ResponseAnalyzerInput
    mindfulness_input: MindfulnessInput
    cognitive_behavioral_input: CognitiveBehavioralInput
    positive_psychology_input: PositivePsychologyInput
    stress_management_input: StressManagementInput
    ethical_input: EthicalInput


class CombinedOutput(BaseModel):
    assesment_output: AssessmentOutput
    response_analyzer_output: ResponseAnalyzerOutput
    mindfulness_output: MindfulnessOutput
    cognitive_behavioral_output: CognitiveBehavioralOutput
    positive_psychology_output: PositivePsychologyOutput
    stress_management_output: StressManagementOutput
    ethical_output: EthicalOutput


AGENT_CONFIG = {
    "ASSESSMENT_AGENT": {
        "name": "Assessment Agent",
        "instructions": "Evaluate the psychological state of the user based on the input.",
        "model": LLM_MODEL_GPT_4_1_NANO,
        "input_type": AssessmentInput,
        "output_type": AssessmentOutput,
        "verbose": GLOBAL_VERBOSE_ENABLED,
    },
    "RESPONSE_ANALYZER_AGENT": {
        "name": "Response Analyzer Agent",
        "instructions": "Analyze the responses and provide a summary and recommendations.",
        "model": LLM_MODEL_GPT_4_1_NANO,
        "input_type": ResponseAnalyzerInput,
        "output_type": ResponseAnalyzerOutput,
        "verbose": GLOBAL_VERBOSE_ENABLED,
    },
    "MINDFULNESS_AGENT": {
        "name": "Mindfulness Agent",
        "instructions": "Provide mindfulness exercises and techniques.",
        "model": LLM_MODEL_GPT_4_1,
        "input_type": MindfulnessInput,
        "output_type": MindfulnessOutput,
        "verbose": GLOBAL_VERBOSE_ENABLED,
    },
    "COGNITIVE_BEHAVIORAL_AGENT": {
        "name": "Cognitive Behavioral Agent",
        "instructions": "Provide cognitive behavioral therapy techniques.",
        "model": LLM_MODEL_GPT_4_1,
        "input_type": CognitiveBehavioralInput,
        "output_type": CognitiveBehavioralOutput,
        "verbose": GLOBAL_VERBOSE_ENABLED,
    },
    "POSITIVE_PSYCHOLOGY_AGENT": {
        "name": "Positive Psychology Agent",
        "instructions": "Provide positive psychology techniques.",
        "model": LLM_MODEL_GPT_4_1,
        "input_type": PositivePsychologyInput,
        "output_type": PositivePsychologyOutput,
        "verbose": GLOBAL_VERBOSE_ENABLED,
    },
    "STRESS_MANAGEMENT_AGENT": {
        "name": "Stress Management Agent",
        "instructions": "Provide stress management techniques.",
        "model": LLM_MODEL_GPT_4_1,
        "input_type": StressManagementInput,
        "output_type": StressManagementOutput,
        "verbose": GLOBAL_VERBOSE_ENABLED,
    },
    "ETHICAL_AGENT": {
        "name": "Ethical Agent",
        "instructions": "Provide ethical guidelines and considerations.",
        "model": LLM_MODEL_GPT_4_1,
        "input_type": EthicalInput,
        "output_type": EthicalOutput,
        "verbose": GLOBAL_VERBOSE_ENABLED,
    },
}

## Step 6: Instantiate All Psychometric Agents

This step uses the `AGENT_CONFIG` dictionary to create a list of all agent instances using the `from_batch_config` method.

Each agent is wrapped in a `PsychometricAgentContainer` with:

- A unique name

- The instantiated `PsychometricsAgent`

These agents can now be accessed and run in sequence or independently.


In [ ]:
agents = PsychometricsAgent.from_batch_config(AGENT_CONFIG)

## 🎯 Conclusion:

Agentic Psychometric Analysis demonstrates the power of combining modular AI agents with structured psychometric evaluation. By leveraging specialized agents for distinct psychological dimensions — ranging from assessment and response analysis to mindfulness, CBT, and ethical considerations — the system delivers a rich, multi-faceted understanding of user inputs.

Its extensible architecture, built on top of OpenAI models and Pydantic validation, allows for easy customization and integration into a variety of applications, including:

- Mental health research  

- Personal development tools  

- AI-driven therapy simulations


This project showcases how thoughtful agent design and a clear interface can enable nuanced, context-aware psychological insights — paving the way for the next generation of empathetic and intelligent mental wellness systems.


---

# Thank You for visiting The Hackers Playbook! 🌐

If you liked this research material;

- [Subscribe to our newsletter.](https://thehackersplaybook.substack.com)

- [Follow us on LinkedIn.](https://www.linkedin.com/company/the-hackers-playbook/)

- [Leave a star on our GitHub.](https://www.github.com/thehackersplaybook)

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
</div>